# 8. Как Graph-EVT-agent проходит путь от данных к источнику

Этот ноутбук **исполняет**, а не имитирует, `GraphEVTPipeline` на многоканальном наборе `kuramoto_synchronized_series.csv`. Он показывает две разные графические сущности:

1. **граф конвейера** — какие этапы выполняются и где находится условное ветвление;
2. **граф зависимостей** — какую сеть каналов агент оценил только по baseline.

Численная часть детерминирована. Здесь нет LLM: слово *agent* обозначает оркестрацию проверки входа, EVT-детекции, построения графа и ранжирования источника. Ранжирование — статистическая гипотеза, а не доказательство причинности.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import numpy as np
import pandas as pd

from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig

SEED = 42
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data/generated/kuramoto_synchronized_series.csv"
if not DATA_PATH.exists():
    DATA_PATH = ROOT / "data/kuramoto_synchronized_series.csv"

dataframe = pd.read_csv(DATA_PATH)
time_column = "timestamp"
channel_names = [column for column in dataframe.columns if column != time_column]
timestamps = dataframe[time_column].to_numpy()
values = dataframe[channel_names].to_numpy()
print(f"data: {values.shape[0]} time points × {values.shape[1]} channels")
dataframe.head()

## 1. Исполняемый граф конвейера

`run_detailed` сначала валидирует форму, временную ось и пропуски. Затем один и тот же робастно нормированный EVT-индикатор строится для всех моментов, но его POT/GPD-порог оценивается **только на baseline**. Одномерные данные или отсутствие тревоги завершают маршрут без графа. Для обнаруженного многоканального события граф также строится только по baseline; после этого каналы ранжируются по раннему и сильному отклику с учётом соседей.

In [ ]:
stages = {
    "input": (0.08, 0.52, "CSV / массив\n[время, канал]"),
    "inspect": (0.27, 0.52, "InputInspector\nсхема • время • NaN"),
    "evt": (0.48, 0.52, "EVT detect\nMAD → top-k → POT/GPD"),
    "decision": (0.67, 0.52, "Событие есть\nи вход n-D?"),
    "graph": (0.82, 0.72, "Граф baseline\nAR / DFA / Kuramoto"),
    "rank": (0.82, 0.32, "Source ranking\nпик • энергия • latency"),
    "stop": (0.67, 0.12, "Стоп\nбез локализации"),
}
edges = [
    ("input", "inspect", ""), ("inspect", "evt", ""), ("evt", "decision", ""),
    ("decision", "graph", "да"), ("graph", "rank", "замороженная сеть"),
    ("decision", "stop", "нет"),
]
fig, ax = plt.subplots(figsize=(13, 5.5))
for key, (x, y, label) in stages.items():
    width, height = (0.16, 0.18)
    color = "#dbeafe" if key not in {"decision", "stop"} else ("#fef3c7" if key == "decision" else "#f3f4f6")
    box = FancyBboxPatch((x-width/2, y-height/2), width, height,
                         boxstyle="round,pad=0.015", facecolor=color,
                         edgecolor="#334155", linewidth=1.5)
    ax.add_patch(box)
    ax.text(x, y, label, ha="center", va="center", fontsize=10)
for start, end, label in edges:
    x1, y1, _ = stages[start]; x2, y2, _ = stages[end]
    arrow = FancyArrowPatch((x1+0.085, y1), (x2-0.085, y2), arrowstyle="-|>",
                            mutation_scale=13, linewidth=1.5, color="#475569",
                            connectionstyle="arc3,rad=0.0")
    if start == "decision":
        arrow.set_positions((x1, y1-0.08 if end == "stop" else y1+0.08),
                            (x2, y2+0.08 if end == "stop" else y2-0.08))
    elif start == "graph" and end == "rank":
        arrow.set_positions((x1, y1-0.09), (x2, y2+0.09))
    ax.add_patch(arrow)
    if label:
        ax.text((x1+x2)/2 + (0.02 if start == "decision" else 0), (y1+y2)/2,
                label, fontsize=9, color="#334155", backgroundcolor="white")
ax.text(0.54, 0.93, "baseline используется для калибровки — будущие точки не обучают порог или граф",
        ha="center", fontsize=10, color="#9f1239")
ax.set(xlim=(0, 0.94), ylim=(0, 1), title="Рабочий маршрут GraphEVTPipeline.run_detailed")
ax.axis("off")
plt.show()

## 2. Запуск на данных

Первые 300 наблюдений объявляются baseline **до просмотра результата**. Метод `ar` строит ребро по абсолютной корреляции инноваций отдельных AR-моделей. `persistence=2` требует два последовательных превышения EVT-порога.

In [ ]:
BASELINE_SIZE = 300
pipeline = GraphEVTPipeline(
    EVTConfig(
        baseline_size=BASELINE_SIZE,
        tail_quantile=0.90,
        alarm_probability=0.99,
        top_k=3,
        persistence=2,
    ),
    graph=GraphConfig(method="ar", edge_threshold=0.35, random_state=SEED),
    input_config=InputConfig(missing="error", require_regular_time=True),
)
result = pipeline.run_detailed(dataframe.set_index(time_column)[channel_names])

summary = pd.Series({
    "route": result.input_profile.kind,
    "time points": result.input_profile.n_time,
    "channels": result.input_profile.n_channels,
    "detected": result.detection.detected,
    "alarm index": result.detection.time_index,
    "alarm threshold": result.detection.alarm_threshold,
    "graph method": None if result.graph is None else result.graph.method,
    "top source": None if result.ranking is None else channel_names[result.ranking.source],
}, name="pipeline output")
summary.to_frame()

## 3. Связь входа, EVT-решения и ответа

На рисунке ниже вертикальная линия отделяет baseline. EVT-индикатор — среднее трёх наибольших абсолютных robust z-score в каждый момент. Первое устойчивое превышение красного порога запускает графовую локализацию. Вероятности справа нормированы внутри эвристического ranking и не являются калиброванными вероятностями физической причины.

In [ ]:
detection = result.detection
fig, axes = plt.subplots(3, 1, figsize=(13, 10), constrained_layout=True)

axes[0].plot(timestamps, values, alpha=0.75, linewidth=0.8)
axes[0].axvline(timestamps[BASELINE_SIZE], color="black", linestyle="--", label="конец baseline")
axes[0].set(title="Входные каналы", ylabel="значение")
axes[0].legend(loc="upper left", ncol=2)

axes[1].plot(timestamps, detection.indicator, color="#2563eb", label="EVT-индикатор")
axes[1].axhline(detection.alarm_threshold, color="#dc2626", linestyle="--", label="POT/GPD alarm threshold")
axes[1].axvspan(timestamps[0], timestamps[BASELINE_SIZE-1], color="#94a3b8", alpha=0.18, label="baseline")
if detection.time_index is not None:
    axes[1].axvline(timestamps[detection.time_index], color="#7c3aed", linewidth=2, label="первая тревога")
axes[1].set(title="Решение EVT", ylabel="robust score")
axes[1].legend(loc="upper left", ncol=4)

if result.ranking is not None:
    probabilities_by_node = np.zeros(len(channel_names))
    probabilities_by_node[result.ranking.node_ids] = result.ranking.probabilities
    colors = ["#dc2626" if node == result.ranking.source else "#60a5fa" for node in range(len(channel_names))]
    axes[2].bar(channel_names, probabilities_by_node, color=colors)
    axes[2].set(title="Выход: ранжирование предполагаемого начального источника", ylabel="ranking probability")
else:
    axes[2].text(0.5, 0.5, "Локализация пропущена: нет обнаруженного n-D события", ha="center", va="center")
    axes[2].set_axis_off()
plt.show()

## 4. Оценённый граф зависимостей

Узел — канал, линия — ребро, прошедшее порог зависимости на baseline. Толщина и непрозрачность кодируют оценённую силу связи. Красный узел — первый в ranking. Самосвязи не рисуются. Этот граф описывает статистическую зависимость baseline и не задаёт направление распространения.

In [ ]:
if result.graph is None:
    print("Граф не построен: pipeline строит его только для обнаруженного многоканального события.")
else:
    adjacency = result.graph.adjacency
    dependence = result.graph.dependence
    node_count = len(channel_names)
    angles = np.linspace(0, 2*np.pi, node_count, endpoint=False) + np.pi/2
    positions = np.column_stack([np.cos(angles), np.sin(angles)])
    fig, ax = plt.subplots(figsize=(8, 8))
    for left in range(node_count):
        for right in range(left + 1, node_count):
            if adjacency[left, right]:
                strength = float(dependence[left, right])
                ax.plot(*positions[[left, right]].T, color="#64748b",
                        linewidth=0.5 + 4*strength, alpha=0.25 + 0.65*strength, zorder=1)
    node_colors = ["#ef4444" if node == result.ranking.source else "#93c5fd" for node in range(node_count)]
    ax.scatter(positions[:, 0], positions[:, 1], s=1100, c=node_colors,
               edgecolor="#1e293b", linewidth=1.5, zorder=2)
    for node, name in enumerate(channel_names):
        ax.text(*positions[node], name, ha="center", va="center", fontsize=9, zorder=3)
    edge_count = int(np.triu(adjacency, 1).sum())
    ax.set_title(f"Baseline graph: {result.graph.method} • {edge_count} undirected edges")
    ax.set_aspect("equal")
    ax.axis("off")
    plt.show()

## Как интерпретировать результат

- **Обнаружение:** утверждает, что после baseline возникло устойчивое экстремальное значение агрегированного индикатора относительно fitted tail.
- **Граф:** фиксирует зависимости, оценённые до события; этим ограничивается утечка информации из будущего.
- **Локализация:** сочетает пик, энергию, задержку, степень и отклик соседей. Это объяснимая эвристика, а не обученная causal-модель.
- **Ветвление:** для одномерного входа доступны профиль и EVT; для n-D входа без тревоги граф и ranking намеренно равны `None`.
- **Проверка устойчивости:** перед содержательным выводом варьируйте baseline, tail quantile, alarm probability, persistence и метод графа; согласие одного запуска недостаточно.